## OpenAlex Crawling

* Sector: `Transportation` (for. Pilot test)

### Import Libraries

In [1]:
import csv
import json
import time
from pathlib import Path

import requests

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

### Configurations

In [11]:
BASE_URL = "https://api.openalex.org/works"
RATE_LIMIT_URL = "https://api.openalex.org/rate-limit"

PER_PAGE = 100
CONNECT_TIMEOUT = 10
READ_TIMEOUT = 30 
MAX_RETRIES = 5
REQUEST_DELAY = 0.5
QUERY_DELAY = 2
MAX_LIMIT_WAIT_SECONDS = 300

In [3]:
SELECT_FIELDS = ",".join([
    "id",
    "title",
    "doi",
    "publication_year",
    "cited_by_count",
    "abstract_inverted_index",
    "authorships",
    "primary_location",
])

CSV_COLUMNS = [
    "query",
    "title",
    "doi",
    "publication_year",
    "journal",
    "authors",
    "abstract",
    "openalex_id",
    "cited_by_count",
]

### Function Definition

* `load_keywords()`: Loads search keywords(query) from 'keywords.txt'.
  - Input: your keyword file path
* `parse_abstract()`: Reconstructs plain abstract text from OpenAlex inverted index format.
* `get_journal_name()`: Extracts the journal or source name from one OpenAlex work record.
* `normalize_work()`: Converts one raw API result into one flat CSV row.
  - Extract title, DOI number, publication year, journal, authors, ... etc.
  - Standardizes missing values.

In [13]:
# Keyword loading from text file
def load_keywords(file_path: Path) -> list[str]:
    if not file_path.exists():
        raise FileNotFoundError(f"Keyword file not found: {file_path}")

    keywords = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            value = line.strip()
            if value and not value.startswith("#"):
                keywords.append(value)

    return keywords

In [14]:
# parsing text (abstract, journal name, title, doik, publication year,authors, OpenAlex ID, and citation counts)
def parse_abstract(abstract_inverted_index: dict | None) -> str:
    if not abstract_inverted_index:
        return ""

    try:
        max_position = max(
            max(position_list) for position_list in abstract_inverted_index.values()
        )
        words = [""] * (max_position + 1)

        for word, position_list in abstract_inverted_index.items():
            for position in position_list:
                words[position] = word

        return " ".join(word for word in words if word)
    except Exception:
        return ""


def get_journal_name(work: dict) -> str:
    primary_location = work.get("primary_location") or {}
    source = primary_location.get("source") or {}
    return source.get("display_name", "")


def normalize_work(work: dict, query: str) -> dict:
    authors = []
    for item in work.get("authorships", []):
        author = item.get("author") or {}
        name = author.get("display_name", "").strip()
        if name:
            authors.append(name)

    return {
        "query": query,
        "title": work.get("title", "") or "",
        "doi": work.get("doi", "") or "",
        "publication_year": work.get("publication_year", "") or "",
        "journal": get_journal_name(work),
        "authors": ", ".join(authors),
        "abstract": parse_abstract(work.get("abstract_inverted_index")),
        "openalex_id": work.get("id", "") or "",
        "cited_by_count": work.get("cited_by_count", 0) or 0,
    }

* `build_session()`: Creates a reusable HTTP session for API requiests.
    - Input: None
    - Sets a custom **User-Agent**.
    - Adds `mailto` information <span style="color:red">if email is available</span>.
* `build_params()`: Builds query parameters for one OpenAlex requests.
    - Input: keywords(str), cursor(str)
    - Adds paging and selected fields
    - Uses either legacy `title_and_abstract.search` or current `search`.
    - Adds `api_key` and `mailto` <span style="color:red">if available</span>.
* `get_rate_limit_status()`: Requests current daily budget information from OpenAlex.
    - Returns budget information **if api key exists**.
    - Returns **None** if no key is available or the requests fails.
* `handle_limit()`: Handles rate-limit responses.
    - Waits before retrying.
      + If an api key exists, checks whether the daily free budget is exhausted.
      + Raises an error if the daily budget is fully used.
* `fetch_page()`: Fetches one page of results.
    - Sends one paginated API request.
    - Retries temporary failures.
      + `results`: current page results
      + `next_cursor`: cursor for the next page

In [15]:
#Setting parameters and sessions
# def build_session(email: str = "") -> requests.Session:
#     session = requests.Session()

#     user_agent = "OpenAlexCollector/1.0"
#     if email:
#         user_agent = f"{user_agent} (mailto:{email})"

#     session.headers.update({"User-Agent": user_agent})
#     return session

def build_session() -> requests.Session:
    session = requests.Session()

    # MODIFIED: email/mailto removed; key-only use is the primary path now
    session.headers.update({"User-Agent": "OpenAlexCollector/1.0"})
    return session


def build_params(
    keyword: str,
    cursor: str,
    year_filter: str,
    api_key: str = "",
    # email: str = "",     # MODIFIED: email/mailto removed
    use_legacy_title_abstract_search: bool = True,
) -> dict:
    params = {
        "per_page": PER_PAGE,
        "cursor": cursor,
        "select": SELECT_FIELDS,
    }

    if use_legacy_title_abstract_search:
        params["filter"] = f"title_and_abstract.search:{keyword},{year_filter}"
    else:
        params["search"] = keyword
        params["filter"] = year_filter

    if api_key:
        params["api_key"] = api_key

    # if email:
    #     params["mailto"] = email     # MODIFIED: email/mailto removed

    return params

* `initialize_output()`: Creates the combined output CSV if it does not exist.
    - Creates the output directory if needed.
    - Writes the CSV header once.
* `load_existing_ids()`: Loads already saved OpenAlex IDs from the output file.
    - Used for deduplications.
* `load_checkpoint()`: Loads completed query names from the checkpoint file.
    - Saves completed query names to disk.
    - Writes completed queries as JSON.
* `log_failed_query()`: Records failed queries in a log file.
    - Appends one failed query per line.
* `append_rows()`: Appends collected rows to the combined CSV file.
    - Writes only new rows.
* `collect_keywords()`: Collects all pages for one keyword.
    - Repeats API requests until no ntext cursor exists.
    - Deduplicates by `openalex_id`.

In [16]:
#Rate-limit and budget check
def get_rate_limit_status(session: requests.Session, api_key: str = "") -> dict | None:
    if not api_key:
        return None

    try:
        response = session.get(
            RATE_LIMIT_URL,
            params={"api_key": api_key},
            timeout=(CONNECT_TIMEOUT, READ_TIMEOUT),
        )
        response.raise_for_status()
        payload = response.json()
        return payload.get("rate_limit", payload)
    except Exception:
        return None


def handle_limit(
    session: requests.Session,
    response: requests.Response,
    attempt: int,
    api_key: str = "",
) -> None:
    # MODIFIED: stop extremely long waits instead of sleeping for hours
    retry_after = response.headers.get("Retry-After")
    wait_time = int(retry_after) if retry_after and retry_after.isdigit() else 30 * attempt

    rate_info = get_rate_limit_status(session, api_key)
    if rate_info:
        remaining = rate_info.get("daily_remaining_usd")
        reset_in = rate_info.get("resets_in_seconds")

        if remaining == 0:
            raise RuntimeError(
                f"Daily free budget exhausted (daily_remaining_usd=0, resets_in_seconds={reset_in})"
            )

        if wait_time > MAX_LIMIT_WAIT_SECONDS:
            raise RuntimeError(
                f"Request limit wait is too long ({wait_time}s). Stop and retry later. "
                f"daily_remaining_usd={remaining}, resets_in_seconds={reset_in}"
            )

        print(
            f"Request limit reached. Waiting {wait_time}s "
            f"(daily_remaining_usd={remaining}, resets_in_seconds={reset_in})",
            flush=True,
        )
    else:
        if wait_time > MAX_LIMIT_WAIT_SECONDS:
            raise RuntimeError(
                f"Request limit wait is too long ({wait_time}s). Stop and retry later."
            )

        print(f"Request limit reached. Waiting {wait_time}s", flush=True)

    time.sleep(wait_time)



def fetch_page(
    session: requests.Session,
    keyword: str,
    cursor: str,
    page_number: int,  # MODIFIED: explicit page number for tracking
    year_filter: str,
    api_key: str = "",
    use_legacy_title_abstract_search: bool = True,
) -> tuple[list[dict], str | None]:
    params = build_params(
        keyword=keyword,
        cursor=cursor,
        year_filter=year_filter,
        api_key=api_key,
        use_legacy_title_abstract_search=use_legacy_title_abstract_search,
    )

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            # MODIFIED: minimal page-level tracking
            print(f"Requesting page {page_number} (attempt {attempt}/{MAX_RETRIES})", flush=True)

            response = session.get(
                BASE_URL,
                params=params,
                timeout=(CONNECT_TIMEOUT, READ_TIMEOUT),
            )

            if response.status_code == 429:
                handle_limit(session, response, attempt, api_key)
                continue

            response.raise_for_status()
            payload = response.json()

            results = payload.get("results", [])
            next_cursor = payload.get("meta", {}).get("next_cursor")

            return results, next_cursor

        except requests.exceptions.RequestException as e:
            if attempt == MAX_RETRIES:
                raise RuntimeError(f"Request failed after {MAX_RETRIES} retries: {e}") from e

            wait_time = 10 * attempt
            print(f"Request failed. Waiting {wait_time}s ({attempt}/{MAX_RETRIES})", flush=True)
            time.sleep(wait_time)

    return [], None

In [17]:
#preparing CSV file to export 
def initialize_output(output_dir: Path, output_file: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    if not output_file.exists():
        with open(output_file, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
            writer.writeheader()


def load_existing_ids(output_file: Path) -> set[str]:
    existing_ids = set()

    if not output_file.exists():
        return existing_ids

    with open(output_file, "r", newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            openalex_id = (row.get("openalex_id") or "").strip()
            if openalex_id:
                existing_ids.add(openalex_id)

    return existing_ids


def load_checkpoint(checkpoint_file: Path) -> dict:  # MODIFIED
    if not checkpoint_file.exists():
        return {
            "done_queries": [],
            "query_state": {}
        }

    with open(checkpoint_file, "r", encoding="utf-8") as f:
        return json.load(f)


def save_checkpoint(checkpoint: dict, checkpoint_file: Path) -> None:  # MODIFIED
    checkpoint_file.parent.mkdir(parents=True, exist_ok=True)

    with open(checkpoint_file, "w", encoding="utf-8") as f:
        json.dump(checkpoint, f, ensure_ascii=False, indent=2)


def append_rows(rows: list[dict], output_file: Path) -> None:
    if not rows:
        return

    with open(output_file, "a", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        writer.writerows(rows)


def collect_keyword(
    session: requests.Session,
    keyword: str,
    seen_ids: set[str],
    output_file: Path,
    year_filter: str,
    api_key: str = "",
    use_legacy_title_abstract_search: bool = True,
    start_cursor: str = "*",              
    start_page_count: int = 0,            
    start_total_new_rows: int = 0,        
    checkpoint: dict | None = None,       
    checkpoint_file: Path | None = None,  
) -> tuple[int, int]:
    cursor = start_cursor                 
    total_new_rows = start_total_new_rows 
    page_count = start_page_count         

    while cursor:
        current_page = page_count + 1  # MODIFIED: page tracking

        results, next_cursor = fetch_page(
            session=session,
            keyword=keyword,
            cursor=cursor,
            page_number=current_page,
            year_filter=year_filter,
            api_key=api_key,
            use_legacy_title_abstract_search=use_legacy_title_abstract_search,
        )

        if not results:
            print(f"No results returned on page {current_page}", flush=True)
            break

        rows_to_save = []

        for work in results:
            row = normalize_work(work, keyword)
            openalex_id = row["openalex_id"]

            if not openalex_id or openalex_id in seen_ids:  
                continue

            seen_ids.add(openalex_id)
            rows_to_save.append(row)

        append_rows(rows_to_save, output_file)

        total_new_rows += len(rows_to_save)
        page_count += 1
        cursor = next_cursor

        if checkpoint is not None and checkpoint_file is not None:  # MODIFIED
            checkpoint["query_state"][keyword] = {
                "cursor": cursor,
                "page_count": page_count,
                "total_new_rows": total_new_rows,
            }
            save_checkpoint(checkpoint, checkpoint_file)

        print(
            f"Received page {current_page}: "
            f"results={len(results)}, new_rows={len(rows_to_save)}, total_new_rows={total_new_rows}",
            flush=True,
        )
        time.sleep(REQUEST_DELAY)

    return total_new_rows, page_count

In [18]:
def run_collection(
    keyword_file: str = "keywords.txt",
    output_dir: str = "openalex_output",
    output_name: str = "transport_combined.csv",
    year_filter: str = "from_publication_date:2015-01-01",
    api_key: str = "",
    use_legacy_title_abstract_search: bool = True,
) -> None:
    keyword_path = Path(keyword_file)
    output_dir_path = Path(output_dir)
    output_file = output_dir_path / output_name
    checkpoint_file = output_dir_path / "checkpoint.json"
    failed_log_file = output_dir_path / "failed_queries.txt"

    keywords = load_keywords(keyword_path)
    initialize_output(output_dir_path, output_file)

    session = build_session()

    checkpoint = load_checkpoint(checkpoint_file)              # MODIFIED
    done_queries = set(checkpoint.get("done_queries", []))    # MODIFIED
    query_state = checkpoint.get("query_state", {})           # MODIFIED

    seen_ids = load_existing_ids(output_file)

    if api_key:
        rate_info = get_rate_limit_status(session, api_key)
        if rate_info:
            print(
                "Rate limit status: "
                f"daily_budget_usd={rate_info.get('daily_budget_usd')}, "
                f"daily_used_usd={rate_info.get('daily_used_usd')}, "
                f"daily_remaining_usd={rate_info.get('daily_remaining_usd')}, "
                f"resets_in_seconds={rate_info.get('resets_in_seconds')}",
                flush=True,
            )

    iterator = keywords
    if tqdm is not None:
        iterator = tqdm(keywords, desc="Queries", unit="query")

    for index, keyword in enumerate(iterator, start=1):
        if keyword in done_queries:
            print(f"Skipping [{index}/{len(keywords)}]: {keyword}", flush=True)
            continue

        # MODIFIED: load saved resume state for this keyword
        state = query_state.get(keyword, {})
        start_cursor = state.get("cursor", "*")
        start_page_count = state.get("page_count", 0)
        start_total_new_rows = state.get("total_new_rows", 0)

        print(f"Collecting [{index}/{len(keywords)}]: {keyword}", flush=True)

        try:
            new_rows, pages = collect_keyword(
                session = session,
                keyword = keyword,
                seen_ids = seen_ids,
                output_file = output_file,
                year_filter = year_filter,
                api_key = api_key,
                use_legacy_title_abstract_search = use_legacy_title_abstract_search,
                start_cursor = start_cursor,                     # MODIFIED
                start_page_count = start_page_count,             # MODIFIED
                start_total_new_rows = start_total_new_rows,     # MODIFIED
                checkpoint = checkpoint,                         # MODIFIED
                checkpoint_file = checkpoint_file,               # MODIFIED
            )

            done_queries.add(keyword)
            checkpoint["done_queries"] = sorted(done_queries)   # MODIFIED
            checkpoint["query_state"].pop(keyword, None)        # MODIFIED
            save_checkpoint(checkpoint, checkpoint_file)        # MODIFIED

            print(
                f"Completed [{index}/{len(keywords)}]: pages={pages}, new_rows={new_rows}",
                flush=True,
            )

        except Exception as e:
            log_failed_query(keyword, str(e), failed_log_file)
            print(f"Failed [{index}/{len(keywords)}]: {keyword} ({e})", flush=True)

        time.sleep(QUERY_DELAY)

    print("Collection finished", flush=True)
    print(f"Output file: {output_file}", flush=True)
    print(f"Unique ids: {len(seen_ids)}", flush=True)
    print(f"Completed queries: {len(done_queries)}", flush=True)

### Just run below code (after running all codes above)
* Change User settings blocks personally.
* `USE_LEGACY_TITLE_ABSTRACT_SEARCH` (optional setting)
    - True: `title_and_abstract.search` (include title and abstract)
    - False: `filter=field.search` (include title, abstract, and full-text)
    - For related information, please visit developers.openalex.org

In [64]:
# User settings (you just change this settings personally)
API_KEY = "MiD4QVyoNrVenxKUFugfmF" #your api key (if you do not have api key, you can skip this)
#EMAIL = "" #your email id (For safe running, if you enter your api key, you better skip enter email address)

KEYWORD_FILE = "keywords.txt" #your keyword file
OUTPUT_DIR = "." #directory where you save the file
OUTPUT_NAME = "Transportation_pilot.csv" #file name (.csv)
YEAR_FILTER = "from_publication_date:2015-01-01" #publication year setting


# True: behavior close to the original code
# False: use the current recommended "search" parameter
USE_LEGACY_TITLE_ABSTRACT_SEARCH = True

run_collection(
    keyword_file = KEYWORD_FILE,
    output_dir = OUTPUT_DIR,
    output_name = OUTPUT_NAME,
    year_filter = YEAR_FILTER,
    api_key = API_KEY,
    #email = EMAIL,
    use_legacy_title_abstract_search = USE_LEGACY_TITLE_ABSTRACT_SEARCH,
)

## Interrupt

Rate limit status: daily_budget_usd=1, daily_used_usd=0, daily_remaining_usd=1, resets_in_seconds=80736


Queries:   0%|          | 0/17 [00:00<?, ?query/s]

Skipping [1/17]: Electric Vehicle
Skipping [2/17]: EV
Skipping [3/17]: Hybrid Electric Vehicle
Skipping [4/17]: HEV
Skipping [5/17]: Autonomous Vehicle
Skipping [6/17]: AV
Skipping [7/17]: In-Vehicle Infotainment
Skipping [8/17]: IVI
Skipping [9/17]: Shared Mobility
Skipping [10/17]: Shared Mobility Service
Skipping [11/17]: Smart Ship
Skipping [12/17]: Drone
Skipping [13/17]: Service Robot
Skipping [14/17]: Smart Logistics
Skipping [15/17]: Secondary Battery
Skipping [16/17]: Intelligent Transportation Systems
Requesting page 29059 (attempt 1/5)
Received page 29059: results=100, new_rows=59, total_new_rows=2286170
Requesting page 29060 (attempt 1/5)
Received page 29060: results=100, new_rows=96, total_new_rows=2286266
Requesting page 29061 (attempt 1/5)
Received page 29061: results=100, new_rows=97, total_new_rows=2286363
Requesting page 29062 (attempt 1/5)
Received page 29062: results=100, new_rows=95, total_new_rows=2286458
Requesting page 29063 (attempt 1/5)
Received page 29063: re

KeyboardInterrupt: 

* **Api key list**
1) dOA1XY7pxifLHRUScJRE0J (my key1) -
2) MiD4QVyoNrVenxKUFugfmF (andrew's key) -
3) 4G55echPp6r7crZLMjmEav (my key2) -
4) 8a9YLXuWzgdPxVjgdmaBjZ (my key3) -
5) 11gxmVmVrsT8o9zxLgG4IQ (my key4) -
6) HdEcosPPE4POdEJvzDNOuB (my key5) -
7) x8xTcpW2hZPRlg2c47caFe (my key6) -
8) GjB4tHvD0RaAUlBdja5Nh0 (my key7) -
9) pYxYfs3AlarJF4wbibsf3l (anderw's key2) -
10) gxTfRxl9zDnjvyexT3OK1t (andrew's key3) -working

## Post-processing
* The query keywords are split into two files based on abbreviations and full-text.

In [65]:
import csv
from pathlib import Path

SOURCE_FILE = Path("Transportation_pilot.csv")
OUTPUT_FILE = Path("Transportation_pilot_abbrev_only.csv")

TARGET_QUERIES = {"AV", "HEV", "EV", "ITS", "IVI"}

if not SOURCE_FILE.exists():
    raise FileNotFoundError(f"Source file not found: {SOURCE_FILE}")

row_count = 0

with open(SOURCE_FILE, "r", newline="", encoding="utf-8-sig") as src:
    reader = csv.DictReader(src)
    fieldnames = reader.fieldnames

    if not fieldnames:
        raise ValueError("Source CSV has no header.")

    if "query" not in fieldnames:
        raise ValueError("Source CSV does not contain a 'query' column.")

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8-sig") as out:
        writer = csv.DictWriter(out, fieldnames=fieldnames)
        writer.writeheader()

        for row in reader:
            query = (row.get("query") or "").strip()
            if query in TARGET_QUERIES:
                writer.writerow(row)
                row_count += 1

print(f"Done. Saved {row_count} rows to {OUTPUT_FILE}")

Done. Saved 2687731 rows to Transportation_pilot_abbrev_only.csv


In [66]:
SOURCE_FILE = Path("Transportation_pilot.csv")
OUTPUT_FILE = Path("Transportation_pilot_full_phrase_queries.csv")

TARGET_QUERIES = {
    "Electric Vehicle",
    "Hybrid Electric Vehicle",
    "Autonomous Vehicle",
    "In-Vehicle Infotainment",
    "Shared Mobility",
    "Shared Mobility Service",
    "Smart Ship",
    "Drone",
    "Service Robot",
    "Smart Logistics",
    "Secondary Battery",
    "Intelligent Transportation Systems",
}

if not SOURCE_FILE.exists():
    raise FileNotFoundError(f"Source file not found: {SOURCE_FILE}")

row_count = 0

with open(SOURCE_FILE, "r", newline="", encoding="utf-8-sig") as src:
    reader = csv.DictReader(src)
    fieldnames = reader.fieldnames

    if not fieldnames:
        raise ValueError("Source CSV has no header.")

    if "query" not in fieldnames:
        raise ValueError("Source CSV does not contain a 'query' column.")

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8-sig") as out:
        writer = csv.DictWriter(out, fieldnames=fieldnames)
        writer.writeheader()

        for row in reader:
            query = (row.get("query") or "").strip()
            if query in TARGET_QUERIES:
                writer.writerow(row)
                row_count += 1

print(f"Done. Saved {row_count} rows to {OUTPUT_FILE}")

Done. Saved 464323 rows to Transportation_pilot_full_phrase_queries.csv


## Optional
* If the following message is printed <span style="color:blue">[Failed ((Request failed after 5 retries: HTTPSConnectionPool(host='api.openalex.org', port=443): Read timed out. (read timeout=30)) Collection finished)]</span>.
* Then, use this code to check the checkpoint.

In [ ]:
from pathlib import Path
import json

with open(Path(".") / "checkpoint.json", "r", encoding="utf-8") as f:
    checkpoint = json.load(f)

checkpoint.get("query_state", {}).get("keyword") #enter the failed query keyword

## Optional 2
* If the file is too large, you can split it into smaller files based on query keywords.

In [ ]:
import csv
import re
from pathlib import Path

In [ ]:
SOURCE_FILE = Path("Transportation_pilot.csv")
OUTPUT_DIR = Path("split_by_query")

# If None, split all queries found in the source file.
# If you want only specific queries, use a list like:
# e.g., TARGET_QUERIES = ["EV", "ITS"]
TARGET_QUERIES = None

In [ ]:
def make_safe_filename(text: str) -> str:
    safe = re.sub(r"[^\w\-]+", "_", text.strip())
    safe = re.sub(r"_+", "_", safe).strip("_")
    return safe or "query"


def split_csv_by_query(
    source_file: Path,
    output_dir: Path,
    target_queries: list[str] | None = None,
) -> None:
    if not source_file.exists():
        raise FileNotFoundError(f"Source file not found: {source_file}")

    output_dir.mkdir(parents=True, exist_ok=True)

    writers = {}
    file_handles = {}
    row_counts = {}

    try:
        with open(source_file, "r", newline="", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)
            fieldnames = reader.fieldnames

            if not fieldnames:
                raise ValueError("Source CSV has no header.")

            if "query" not in fieldnames:
                raise ValueError("Source CSV does not contain a 'query' column.")

            for row in reader:
                query = (row.get("query") or "").strip()

                if not query:
                    query = "unknown_query"

                if target_queries is not None and query not in target_queries:
                    continue

                if query not in writers:
                    safe_name = make_safe_filename(query)
                    output_file = output_dir / f"{safe_name}.csv"

                    out_f = open(output_file, "w", newline="", encoding="utf-8-sig")
                    writer = csv.DictWriter(out_f, fieldnames=fieldnames)
                    writer.writeheader()

                    file_handles[query] = out_f
                    writers[query] = writer
                    row_counts[query] = 0

                writers[query].writerow(row)
                row_counts[query] += 1

    finally:
        for fh in file_handles.values():
            fh.close()

    print("Split completed.")
    print(f"Source file: {source_file}")
    print(f"Output directory: {output_dir}")
    print(f"Number of query files created: {len(row_counts)}")

    for query, count in sorted(row_counts.items(), key=lambda x: x[0].lower()):
        print(f"{query}: {count}")

In [ ]:
split_csv_by_query(
    source_file=SOURCE_FILE,
    output_dir=OUTPUT_DIR,
    target_queries=TARGET_QUERIES,
)